In [16]:
import json
import mlflow
import joblib
import onnxmltools
import numpy as np
import pandas as pd
import onnxruntime as ort
from datetime import datetime

from lightgbm import LGBMClassifier
from lightgbm import early_stopping
from onnxruntime import InferenceSession
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from onnxmltools.convert.common.data_types import FloatTensorType

import sys
sys.path.append("..")
from utils import get_table, set_mlflow_experiment
from feature_engineering import time_based_split, infer_column_types, bool_to_int, datetime_to_int64, get_feature_names_from_preprocessor, get_feature_names_from_preprocessor

In [2]:
# TODO: creare script main
# TODO: funzione per stampare summary

# 1. Setup

In [49]:
# Model training
train_frac: float = 0.7
val_frac: float = 0.15
drop_importance_below: float = 0.0  # es. 0.0 = niente drop, oppure 1e-6 / 0.0001

# Features
target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"
features_cols = ['chance1x2_quote_diffInitialCurr1',
 'evaluation_valMetrica',
 'chance1x2_quote_diffInitialCurrx',
 'chance1x2_quote_diffInitialCurr2',
 'goalNoGoal_stats_avgGoalAway',
 'goalNoGoal_stats_avgGoalTakenAway',
 'underOver_chance_over15HT',
 'chance1x2_quote_real2',
 'goalNoGoal_stats_avgGoalTakenHome',
 'goalNoGoal_quote_currentGG',
 'goalNoGoal_chance_goal',
 'chance1x2_quote_real1',
 'chance1x2_quote_realx',
 'goalNoGoal_stats_avgGoalHome',
 'evaluation_valScala',
 'goalNoGoal_flashback_goal']

features_cols = ['goalNoGoal_quote_currentGG', 'goalNoGoal_flashback_goal', 'goalNoGoal_chance_goal', 
                 'chance1x2_quote_diffInitialCurrx', 'chance1x2_quote_realx', 'chance1x2_quote_diffInitialCurr1', 
                 'goalNoGoal_stats_avgGoalAway', 'goalNoGoal_stats_avgGoalTakenAway', 'underOver_chance_over15HT', 
                 'chance1x2_quote_real2', 'chance1x2_quote_diffInitialCurr2', 'goalNoGoal_stats_avgGoalHome',
                'goalNoGoal_stats_avgGoalTakenHome', 'evaluation_valMetrica', 'chance1x2_quote_real1', 'evaluation_valScala']

features_cols = ['goalNoGoal_stats_avgGoalTakenHome',
'goalNoGoal_stats_avgGoalTakenAway',
'goalNoGoal_stats_avgGoalHome',
'goalNoGoal_stats_avgGoalAway',
'goalNoGoal_quote_currentGG',
'goalNoGoal_flashback_goal',
'goalNoGoal_chance_goal',
'evaluation_valScala',
'evaluation_valMetrica',
'chance1x2_quote_current2',
'chance1x2_quote_current1']

# Naming
experiment_name: str = f"{target_col}_lightgbm"
run_name: str = experiment_name + f"_{datetime.utcnow():%Y%m%dT%H%M%SZ}"
# LGBM params
early_stopping_rounds: int = 10

lgbm_params = {
    "n_estimators": 8000, # 3000–10000
    "learning_rate": 0.03,
    "num_leaves": 32, # 16-32
    "max_depth": 6, # 4–8
    "min_child_samples": 100, # 50–200
    "min_split_gain": 0.05, # 0.0–0.2
    "subsample": 0.8, # 0.7–0.9 (bagging factor)
    "subsample_freq": 1, # 1
    "colsample_bytree": 0.7, # 0.6–0.9
    "reg_alpha": 0.5, # 0.1–2.0
    "reg_lambda": 10.0, # 5–20
    "objective": "binary",
    "metric": "auc",   # oppure "binary_logloss"
    "random_state": 42,
    "n_jobs": -1,
}


onnx_export_path: str = f"../artifacts/{target_col}/model.onnx"
preprocessor_export_path: str = f"../artifacts/{target_col}/preprocessor.joblib"

# 2. Data Processing

In [50]:
# Set Experiment
set_mlflow_experiment(experiment_name=experiment_name)

# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

In [51]:
# Helpers
def cast_float64(x):
    return x.astype("float64")

def bool_to_int_df(x):
    return bool_to_int(pd.DataFrame(x, columns=bool_cols))

def datetime_to_int64_df(x):
    return datetime_to_int64(pd.DataFrame(x, columns=datetime_cols))


In [52]:
# Feature Engineering 
unuseful_cols = ['team_league', 'team_home', 'team_away']

df = df_loaded.copy()

# Setting odds_col if not present in the data
if odds_col not in df.columns:
    df[odds_col] = 1.5

df = df.drop(unuseful_cols, axis=1)

# Basic sanity
df = df.dropna(how='all', axis=1)
# target must be 0/1
df[target_col] = df[target_col].astype(int)

num_cols, bool_cols, datetime_cols, cat_cols = infer_column_types(df, target_col)

train_df, val_df, test_df = time_based_split(df=df, time_col="time", train_frac=train_frac, val_frac=val_frac)

X_train = train_df[features_cols]
y_train = train_df[target_col].values

X_val = val_df[features_cols]
y_val = val_df[target_col].values

X_test = test_df[features_cols]
y_test = test_df[target_col].values

tot_cols = features_cols + [target_col]
num_cols = [x for x in num_cols if x in tot_cols]
bool_cols = [x for x in bool_cols if x in tot_cols]
datetime_cols = [x for x in datetime_cols if x in tot_cols]
cat_cols = [x for x in cat_cols if x in tot_cols]


preprocessor = ColumnTransformer(
    transformers=[
        ("num", FunctionTransformer(cast_float64, validate=False), num_cols),
        ("bool", FunctionTransformer(bool_to_int_df, validate=False), bool_cols),
        ("dt", FunctionTransformer(datetime_to_int64_df, validate=False), datetime_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=False,
)

model = LGBMClassifier(**lgbm_params)

# 3. Training

In [53]:
mlflow.start_run(run_name=run_name)

# Log split info
mlflow.log_params({
    "train_frac": train_frac,
    "val_frac": val_frac,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    **{f"lgbm__{k}": v for k, v in lgbm_params.items()},
})

preprocessor.fit(X_train)
Xtr = preprocessor.transform(X_train)
Xva = preprocessor.transform(X_val)

# Fit with early stopping using validation
# NB: early_stopping via fit params (LightGBM sklearn API)
model.fit(
    Xtr, y_train,
    eval_set=[(Xva, y_val)],
    eval_metric="auc",
    callbacks=[early_stopping(stopping_rounds=10, verbose=False)],
)

# pipeline sklearn
pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", model),
])

# Predict proba
p_train = pipe.predict_proba(X_train)[:, 1]
p_val = pipe.predict_proba(X_val)[:, 1]
p_test = pipe.predict_proba(X_test)[:, 1]

pred_train = pipe.predict(X_train)
pred_val = pipe.predict(X_val)
pred_test = pipe.predict(X_test)

auc_train = roc_auc_score(y_train, p_train) if len(np.unique(y_train)) > 1 else np.nan
auc_val = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan
auc_test = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else np.nan

precision_train = precision_score(y_train, pred_train)
precision_val = precision_score(y_val, pred_val)
precision_test = precision_score(y_test, pred_test)

mlflow.log_metrics({
    "auc_train": float(auc_train) if np.isfinite(auc_train) else -1.0,
    "auc_val": float(auc_val) if np.isfinite(auc_val) else -1.0,
    "auc_test": float(auc_test) if np.isfinite(auc_test) else -1.0,
})

# Feature importance
prep_fitted = pipe.named_steps["prep"]
feature_names = get_feature_names_from_preprocessor(prep_fitted)

booster = pipe.named_steps["clf"].booster_
importances = booster.feature_importance(importance_type="gain")
imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance_gain": importances
}).sort_values("importance_gain", ascending=False)

imp_csv = f"../artifacts/{target_col}/feature_importance_gain.csv"
imp_df.to_csv(imp_csv)
mlflow.log_artifact(imp_csv)

# Optional drop features under threshold & retrain
if drop_importance_below > 0.0:
    keep_mask = imp_df["importance_gain"].values > drop_importance_below
    kept_features = imp_df.loc[keep_mask, "feature"].tolist()
    dropped = int((~keep_mask).sum())
    mlflow.log_params({
        "drop_importance_below": drop_importance_below,
        "dropped_features_count": dropped,
        "kept_features_count": len(kept_features),
    })

    # Per droppare in modo robusto con one-hot: selezioniamo colonne DOPO il preprocessor
    # Strategy: trasformiamo X_* e poi addestriamo un secondo LGBM su matrice ridotta.
    Xtr = prep_fitted.transform(X_train)
    Xva = prep_fitted.transform(X_val)
    Xte = prep_fitted.transform(X_test)

    keep_idx = np.where(keep_mask)[0]
    Xtr_k = Xtr[:, keep_idx]
    Xva_k = Xva[:, keep_idx]
    Xte_k = Xte[:, keep_idx]

    model2 = LGBMClassifier(**lgbm_params)
    model2.fit(
        Xtr_k, y_train,
        eval_set=[(Xva_k, y_val)],
        eval_metric="auc",
    )

    p_val2 = model2.predict_proba(Xva_k)[:, 1]
    p_test2 = model2.predict_proba(Xte_k)[:, 1]
    auc_val2 = roc_auc_score(y_val, p_val2) if len(np.unique(y_val)) > 1 else np.nan
    auc_test2 = roc_auc_score(y_test, p_test2) if len(np.unique(y_test)) > 1 else np.nan

    mlflow.log_metrics({
        "auc_val_dropped": float(auc_val2) if np.isfinite(auc_val2) else -1.0,
        "auc_test_dropped": float(auc_test2) if np.isfinite(auc_test2) else -1.0,
    })

    # Log modello ridotto come artifact “secondario”
    mlflow.lightgbm.log_model(model2, name="lgbm_model_retrained_after_drop")
    # Salviamo anche gli indici keep per riprodurre a runtime
    with open("../artifacts/kept_feature_indices.json", "w") as f:
        json.dump(keep_idx.tolist(), f)
    mlflow.log_artifact("../artifacts/kept_feature_indices.json")

# Log modello pipeline (preprocess + lgbm)
mlflow.sklearn.log_model(pipe, name="sklearn_pipeline_lgbm", input_example=X_train.dropna().iloc[:1])

# ONNX export
Xtr_trans = prep_fitted.transform(X_train)
n_features_trans = Xtr_trans.shape[1]

# Convert LightGBM booster to ONNX
initial_types = [("input", FloatTensorType([None, n_features_trans]))]
onnx_model = onnxmltools.convert_lightgbm(
    booster,
    initial_types=initial_types,
    target_opset=15,
)

with open(onnx_export_path, "wb") as f:
    f.write(onnx_model.SerializeToString())
    
joblib.dump(preprocessor, preprocessor_export_path)
print(f"Preprocessor salvato in {preprocessor_export_path}")

mlflow.log_artifact(onnx_export_path)
mlflow.log_artifact(preprocessor_export_path)

[LightGBM] [Info] Number of positive: 1905, number of negative: 1485
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036537 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1267
[LightGBM] [Info] Number of data points in the train set: 3390, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.561947 -> initscore=0.249067
[LightGBM] [Info] Start training from score 0.249067
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

# 4. Load ONNX Model

In [54]:
# 1. Carica il modello ONNX
sess = ort.InferenceSession(onnx_export_path)

# 2. Carica il preprocessor (su Raspberry Pi)
preprocessor = joblib.load(preprocessor_export_path)

def get_onnx_prediction(sess: InferenceSession, preprocessor: ColumnTransformer, input: pd.DataFrame, threshold: float=0.5):
    # Preprocess input
    x_trans = preprocessor.transform(input)  # trasformazione del preprocessor salvato
    x_trans = x_trans.astype(np.float32)        # ONNX richiede float32
    # ONNX Inference
    input_name = sess.get_inputs()[0].name
    outputs = sess.run(None, {input_name: x_trans})

    # Controlla quanti output ci sono e seleziona probabilità
    y_prob = np.array([x[1] for x in outputs[1]])  # se ONNX produce [label, probability]
    y_pred = np.array([int(x > threshold) for x in y_prob])

    return y_prob, y_pred

y_prob, y_pred = get_onnx_prediction(sess, preprocessor, X_test)

2026-03-03 17:18:23.111077470 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {727} for output label


In [55]:
def check_onnx_predictions(y_prob: np.array, p_test:np.array):
    assert sum(np.round(y_prob,3) == np.round(p_test, 3)) == len(y_prob)

check_onnx_predictions(y_prob, p_test)

# 5. Calculate Treshold and EV

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score

# =========================
# Helpers
# =========================
def roi_from_preds(y_true: np.ndarray, y_pred: np.ndarray, odds: np.ndarray) -> float:
    """
    ROI = profit / stake, stake=1 per bet.
    Profit per bet:
      - se win (y_true=1): (odds - 1)
      - se lose (y_true=0): -1
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    odds = np.asarray(odds).astype(float)

    mask = (y_pred == 1)
    n_bets = int(mask.sum())
    if n_bets == 0:
        return np.nan

    stake = n_bets * 1.0
    profit = np.sum(np.where(y_true[mask] == 1, odds[mask] - 1.0, -1.0))
    return profit / stake


def metrics_at_threshold(p: np.ndarray,
                         y_true: np.ndarray,
                         odds: np.ndarray,
                         thr: float) -> dict:
    y_pred = (p >= thr).astype(int)
    pred_perc = float(y_pred.mean())
    n_bets = int(y_pred.sum())

    prec = precision_score(y_true, y_pred, zero_division=0)

    if n_bets > 0:
        median_odds = float(np.median(np.asarray(odds)[y_pred == 1]))
    else:
        median_odds = np.nan

    roi = roi_from_preds(y_true, y_pred, odds)

    return {
        "thr": thr,
        "roi": roi,
        "precision": prec,
        "median_odds": median_odds,
        "pred_perc": pred_perc,
        "n_bets": n_bets
    }


# =========================
# Probabilità classe positiva
# =========================
p_val = pipe.predict_proba(X_val)[:, 1]
p_test = pipe.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.0, 1.0, 51)

# =========================
# Vincoli
# =========================
n_perc_min = 0.02    # almeno 2% giocate
n_perc_max = None    # es 0.10, oppure None
min_median_odds = None  # es 1.50 oppure None

# =========================
# Odds: devi averle per VAL e TEST
# =========================
odds_val = val_df[odds_col].to_numpy()
odds_test = test_df[odds_col].to_numpy()

# =========================
# Grid search soglia su VAL per massimizzare ROI
# =========================
best = None

for thr in thresholds:
    m = metrics_at_threshold(p_val, y_val, odds_val, thr)

    # Applica vincoli su copertura
    if m["pred_perc"] < n_perc_min:
        continue
    if (n_perc_max is not None) and (m["pred_perc"] > n_perc_max):
        continue

    # Vincolo su median odds (opzionale)
    if (min_median_odds is not None) and (np.isfinite(m["median_odds"])) and (m["median_odds"] < min_median_odds):
        continue

    # Se non ci sono bets ROI è NaN: scarta
    if not np.isfinite(m["roi"]):
        continue

    if best is None:
        best = m
        continue

    # ======= criterio: massimizza ROI
    # tie-breaker: più precisione, poi più copertura (o cambia come preferisci)
    if (m["roi"] > best["roi"]) \
       or (m["roi"] == best["roi"] and m["precision"] > best["precision"]) \
       or (m["roi"] == best["roi"] and m["precision"] == best["precision"] and m["pred_perc"] > best["pred_perc"]):
        best = m

if best is None:
    raise ValueError(
        "Nessuna soglia soddisfa i vincoli. Prova ad abbassare n_perc_min, rimuovere min_median_odds, o aumentare la griglia."
    )

best_thr = best["thr"]

# =========================
# Valuta TEST con la soglia migliore trovata su VAL
# =========================
y_pred_test = (p_test >= best_thr).astype(int)
_, y_pred_test_onnx = get_onnx_prediction(sess, preprocessor, X_test, threshold=best_thr)

# Metriche TEST (pipeline sklearn)
test_roi = roi_from_preds(y_test, y_pred_test, odds_test)
test_prec = precision_score(y_test, y_pred_test, zero_division=0)
test_pred_perc = float(y_pred_test.mean())
test_median_odds = float(np.median(odds_test[y_pred_test == 1])) if y_pred_test.sum() > 0 else np.nan
test_n_bets = int(y_pred_test.sum())

# Metriche TEST (ONNX)
test_roi_onnx = roi_from_preds(y_test, y_pred_test_onnx, odds_test)
test_prec_onnx = precision_score(y_test, y_pred_test_onnx, zero_division=0)
test_pred_perc_onnx = float(np.mean(y_pred_test_onnx))
test_median_odds_onnx = float(np.median(odds_test[np.asarray(y_pred_test_onnx) == 1])) if np.sum(y_pred_test_onnx) > 0 else np.nan
test_n_bets_onnx = int(np.sum(y_pred_test_onnx))


start_date = df['time'].min()
df = df.copy()
df['week'] = ((df['time'] - start_date).dt.days // 7) + 1
weekly_counts = df.groupby('week')[target_col].count()
median_bets_per_week = int(weekly_counts.median())

2026-03-03 17:19:44.763589850 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {727} for output label


In [64]:
summary_filepath = f"../artifacts/{target_col}/summary.txt"

# ---- Validation metrics (best threshold selected by ROI) ----
best_roi = best["roi"]
best_prec = best["precision"]
best_pred_perc = best["pred_perc"]
best_median_odd = best["median_odds"]

val_size = len(p_val)
predicted_val_records = int(round(best_pred_perc * val_size))

# ---- Test metrics (sklearn) ----
test_size = len(p_test)
predicted_test_records = int(round(test_pred_perc * test_size))

# nel tuo report originale usi 'median_odd' e 'roi' come variabili:
median_odd = test_median_odds
roi = test_roi

# ---- ONNX metrics (test) ----
predicted_test_records_onnx = int(round(test_pred_perc_onnx * test_size))

output_str = ""
output_str += "==============================\n"
output_str += "📊 MODEL PERFORMANCE SUMMARY\n"
output_str += "==============================\n\n"

output_str += "🔎 VALIDATION SET (threshold tuned to maximize ROI)\n"
output_str += f"🎯 Best Threshold: {best_thr:.3f}\n"
output_str += f"💰 ROI: {best_roi:.4f}\n"
output_str += f"✅ Precision: {best_prec:.4f}\n"
output_str += (
    f"📈 Predicted Bets: {best_pred_perc*100:.2f}% "
    f"[{predicted_val_records}/{val_size}]\n"
)
output_str += f"Median odd: {best_median_odd:.4f}\n\n"

output_str += "🧪 TEST SET\n"
output_str += f"💰 ROI: {roi:.4f}\n"
output_str += f"✅ Precision: {test_prec:.4f}\n"
output_str += (
    f"📈 Predicted Bets: {test_pred_perc*100:.2f}% "
    f"[{predicted_test_records}/{test_size}]\n"
)
output_str += f"Median odd: {median_odd:.4f}\n\n"

output_str += "⚙️ ONNX MODEL (TEST SET)\n"
output_str += f"💰 ROI: {test_roi_onnx:.4f}\n"
output_str += f"✅ Precision: {test_prec_onnx:.4f}\n"
output_str += (
    f"📈 Predicted Bets: {test_pred_perc_onnx*100:.2f}% "
    f"[{predicted_test_records_onnx}/{test_size}]\n"
)
output_str += f"Median odd: {test_median_odds_onnx:.4f}\n\n"

output_str += "💰 BUSINESS METRICS\n"
output_str += f"📅 Expected Bets per Week: {int(round(median_bets_per_week * test_pred_perc))}\n"
output_str += f"📊 ROI (Test Set): {roi:.4f}\n"

# Salvataggio file
with open(summary_filepath, "w", encoding="utf-8") as f:
    f.write(output_str)

mlflow.log_artifact(summary_filepath)
mlflow.log_metrics({
    "roi_val": float(best_roi) if np.isfinite(best_roi) else -1.0,
    "precision_val": float(best_prec) if np.isfinite(best_prec) else -1.0,
    "median_odd_val": float(best_median_odd) if np.isfinite(best_median_odd) else -1.0,
    "pred_perc_val": float(best_pred_perc) if np.isfinite(best_pred_perc) else -1.0,

    "roi_test": float(roi) if np.isfinite(roi) else -1.0,
    "precision_test": float(test_prec) if np.isfinite(test_prec) else -1.0,
    "median_odd_test": float(median_odd) if np.isfinite(median_odd) else -1.0,
    "pred_perc_test": float(test_pred_perc) if np.isfinite(test_pred_perc) else -1.0,

    "roi_test_onnx": float(test_roi_onnx) if np.isfinite(test_roi_onnx) else -1.0,
    "precision_test_onnx": float(test_prec_onnx) if np.isfinite(test_prec_onnx) else -1.0,
    "median_odd_test_onnx": float(test_median_odds_onnx) if np.isfinite(test_median_odds_onnx) else -1.0,
    "pred_perc_test_onnx": float(test_pred_perc_onnx) if np.isfinite(test_pred_perc_onnx) else -1.0,

    "bets_per_week": int(round(median_bets_per_week * test_pred_perc)) if np.isfinite(test_pred_perc) else -1.0,
})

mlflow.log_params({
    "best_threshold": float(best_thr),
    "n_perc_min": float(n_perc_min),
    "n_perc_max": None if n_perc_max is None else float(n_perc_max),
})

print(output_str)
mlflow.end_run()

📊 MODEL PERFORMANCE SUMMARY

🔎 VALIDATION SET (threshold tuned to maximize ROI)
🎯 Best Threshold: 0.600
💰 ROI: 0.0956
✅ Precision: 0.6886
📈 Predicted Bets: 23.00% [167/726]
Median odd: 1.5700

🧪 TEST SET
💰 ROI: 0.1080
✅ Precision: 0.6890
📈 Predicted Bets: 22.56% [164/727]
Median odd: 1.6100

⚙️ ONNX MODEL (TEST SET)
💰 ROI: 0.1080
✅ Precision: 0.6890
📈 Predicted Bets: 22.56% [164/727]
Median odd: 1.6100

💰 BUSINESS METRICS
📅 Expected Bets per Week: 38
📊 ROI (Test Set): 0.1080

🏃 View run powerful-pig-985 at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/899584078766860/runs/c9bb906d58df4560a509044aeb87637d
🧪 View experiment at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/899584078766860
